# [Traces] T4 + Qwen3-8B + AIME: Validation

## Purpose
Validate strategy tuning from Qwen3-4B transfers to stronger Qwen3-8B model.

## Output
AIME validation problems × 5 samples each = traces for validation

In [ ]:
# Debug: List available datasets/models and install vLLM from offline wheels
import os
import subprocess

print("=== Available Kaggle Input Datasets & Models ===")
if os.path.exists('/kaggle/input'):
    for d in sorted(os.listdir('/kaggle/input')):
        print(f"  /kaggle/input/{d}/")
        subpath = f'/kaggle/input/{d}'
        try:
            for f in os.listdir(subpath)[:10]:
                full = os.path.join(subpath, f)
                if os.path.isdir(full):
                    print(f"    [DIR] {f}/")
                    # Go one level deeper for model directories
                    for sub in os.listdir(full)[:5]:
                        print(f"      - {sub}")
                else:
                    print(f"    - {f}")
        except Exception as e:
            print(f"    Error: {e}")
else:
    print("  /kaggle/input not found!")

# Check specific model paths
model_paths = [
    '/kaggle/input/qwen-3/transformers/qwen3-8b/1',
    '/kaggle/input/qwen-3/transformers/8b/1',
    '/kaggle/input/qwen-3/transformers/default/1',
    '/kaggle/input/qwen-3/pytorch/qwen3-8b/1',
    '/kaggle/input/qwen-3/pytorch/8b/1',
]
print("\n=== Checking Model Paths ===")
for p in model_paths:
    exists = os.path.exists(p)
    print(f"  {p}: {'EXISTS' if exists else 'NOT FOUND'}")
    if exists:
        print(f"    Contents: {os.listdir(p)[:5]}")

# Find wheels directory
wheels_paths = [
    '/kaggle/input/vllm-wheels-py312-cu129/wheels',
    '/kaggle/input/vllm-wheels-py312-cu129',
]

wheels_dir = None
for p in wheels_paths:
    if os.path.exists(p):
        whl_files = [f for f in os.listdir(p) if f.endswith('.whl')]
        if whl_files:
            print(f"\n=== Found {len(whl_files)} wheels at: {p} ===")
            wheels_dir = p
            break

if wheels_dir:
    print(f"Installing vLLM from {wheels_dir}...")
    result = subprocess.run(
        ['pip', 'install', '--no-index', '--find-links', wheels_dir, 'vllm'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("vLLM installed successfully!")
    else:
        print(f"Install failed: {result.stderr[-500:]}")
else:
    print("\n!!! No wheels found!")

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, sys, json, re, math, time, subprocess, tempfile
from collections import Counter
from typing import Optional, Dict, List, Any
import pandas as pd

os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [ ]:
class CFG:
    # Model - Qwen3-8B (stronger model for validation)
    model_name = 'Qwen/Qwen3-8B'
    model_path = '/kaggle/input/qwen-3/transformers/qwen3-8b/1'
    
    system_prompt = (
        'You are a math problem solver. '
        'The final answer must be a non-negative integer. '
        'Place the final integer answer inside \\boxed{}.'
    )
    
    n_samples = 5
    max_turns = 8
    max_tokens = 4096
    temperature = 0.8
    top_p = 0.95
    
    # T4 settings (16GB VRAM) - lower utilization for 8B model
    gpu_memory_utilization = 0.92
    max_model_len = 6144  # Reduced for 8B model on T4
    
    problem_timeout = 240
    code_timeout = 10
    
    output_dir = '/kaggle/working/traces'
    seed = 42

print(f"Config: {CFG.model_name}, {CFG.n_samples} samples, {CFG.max_turns} turns")

In [ ]:
def load_aime_problems():
    """Load AIME problems from Kaggle dataset with robust column detection."""
    path = '/kaggle/input/aime-problem-set-1983-2024/AIME_Dataset_1983_2024.csv'
    df = pd.read_csv(path)
    
    print(f"Loaded {len(df)} AIME problems")
    print(f"Columns: {list(df.columns)}")
    
    # Standardize column names
    df.columns = df.columns.str.lower().str.strip()
    
    # Find problem text column - try multiple patterns
    prob_col = None
    for col in df.columns:
        col_lower = col.lower()
        if col_lower == 'question':
            prob_col = col
            break
        if col_lower == 'problem':
            prob_col = col
            break
    
    if prob_col is None:
        # Look for columns containing 'problem' or 'question' in name
        for col in df.columns:
            if 'question' in col.lower() or ('problem' in col.lower() and 'number' not in col.lower()):
                prob_col = col
                break
    
    if prob_col is None:
        # Last resort: find longest text column
        text_cols = [c for c in df.columns if df[c].dtype == 'object']
        if text_cols:
            avg_lens = {c: df[c].astype(str).str.len().mean() for c in text_cols}
            prob_col = max(avg_lens, key=avg_lens.get)
            print(f"Using longest text column: {prob_col}")
    
    if prob_col is None:
        raise ValueError(f"Cannot find problem column. Available: {list(df.columns)}")
    
    # Rename to standard 'problem' column
    df = df.rename(columns={prob_col: 'problem'})
    print(f"Using '{prob_col}' as problem column")
    
    # Create unique ID if not present
    if 'id' not in df.columns:
        if 'year' in df.columns and 'problem number' in df.columns:
            df['id'] = df['year'].astype(str) + '_' + df['problem number'].astype(str)
        else:
            df['id'] = [f'aime_{i}' for i in range(len(df))]
    
    if 'year' in df.columns:
        print(f"Years: {df['year'].min()} - {df['year'].max()}")
    
    return df

df = load_aime_problems()
print(f"\nFirst problem: {df.iloc[0]['problem'][:100]}...")

In [ ]:
def execute_code(code: str, timeout: int = 10) -> str:
    full_code = "import math, itertools, functools\nfrom fractions import Fraction\n" + code
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(full_code)
            f.flush()
            result = subprocess.run(['python', f.name], capture_output=True, text=True, timeout=timeout)
            os.unlink(f.name)
            if result.returncode != 0:
                return f"[ERROR] {result.stderr[:500]}"
            return result.stdout[:2000] or "[No output]"
    except subprocess.TimeoutExpired:
        return "[ERROR] Timeout"
    except Exception as e:
        return f"[ERROR] {str(e)[:200]}"

print(execute_code("print(2**10)"))

In [ ]:
from vllm import LLM, SamplingParams

model_path = CFG.model_path if os.path.exists(CFG.model_path) else CFG.model_name
print(f"Loading: {model_path}")

llm = LLM(
    model=model_path,
    gpu_memory_utilization=CFG.gpu_memory_utilization,
    max_model_len=CFG.max_model_len,
    trust_remote_code=True,
    seed=CFG.seed,
)
print("Model loaded")

In [ ]:
def extract_answer(text: str) -> Optional[int]:
    for pattern in [r'\\boxed\s*\{\s*([0-9,]+)\s*\}', r'answer\s*(?:is|=)\s*([0-9,]+)']:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            try:
                val = int(matches[-1].replace(',', ''))
                if 0 <= val <= 999:  # AIME range
                    return val
            except: pass
    return None

def extract_code_blocks(text: str) -> List[str]:
    return re.findall(r'```(?:python)?\s*\n(.*?)```', text, re.DOTALL | re.IGNORECASE)

def compute_entropy(logprobs: List[Dict]) -> float:
    if not logprobs: return float('inf')
    total, count = 0.0, 0
    for lp_dict in logprobs:
        if isinstance(lp_dict, dict) and lp_dict:
            ent = sum(-math.exp(lp) * math.log2(max(math.exp(lp), 1e-10)) for lp in lp_dict.values() if lp is not None)
            total += ent
            count += 1
    return total / count if count else float('inf')

In [ ]:
def solve_once(problem_text: str, seed: int) -> Dict[str, Any]:
    messages = [{"role": "system", "content": CFG.system_prompt}, {"role": "user", "content": problem_text}]
    tokenizer = llm.get_tokenizer()
    
    all_logprobs, code_executions = [], []
    answer, answer_source, last_code_output = None, None, None
    turns_used, total_tokens = 0, 0
    
    sampling_params = SamplingParams(temperature=CFG.temperature, top_p=CFG.top_p, max_tokens=CFG.max_tokens, seed=seed, logprobs=5)
    
    for turn in range(CFG.max_turns):
        turns_used = turn + 1
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        outputs = llm.generate([prompt], sampling_params)
        response = outputs[0].outputs[0]
        text = response.text
        total_tokens += len(response.token_ids)
        
        if response.logprobs:
            for lp in response.logprobs:
                if lp: all_logprobs.append({k: v.logprob for k, v in lp.items()})
        
        messages.append({"role": "assistant", "content": text})
        
        ans = extract_answer(text)
        if ans is not None:
            answer, answer_source = ans, "boxed"
            break
        
        code_blocks = extract_code_blocks(text)
        if code_blocks:
            outputs_list = []
            for code in code_blocks:
                output = execute_code(code, CFG.code_timeout)
                is_error = '[ERROR]' in output
                if not is_error: last_code_output = output
                outputs_list.append(output)
                code_executions.append({'turn': turn, 'code': code[:1000], 'output': output[:1000], 'is_error': is_error})
            messages.append({"role": "user", "content": f"Code output:\n```\n{''.join(outputs_list)}\n```\nContinue. Answer in \\boxed{{}}." })
        else:
            messages.append({"role": "user", "content": "Continue. Put answer in \\boxed{}."})
    
    if answer is None and last_code_output:
        for num in re.findall(r'\b(\d{1,3})\b', last_code_output):
            val = int(num)
            if 0 <= val <= 999:
                answer, answer_source = val, "code_fallback"
                break
    
    entropy = compute_entropy(all_logprobs)
    if answer_source == "code_fallback": entropy = max(entropy, 8.0)
    
    return {'answer': answer, 'answer_source': answer_source, 'entropy': entropy, 'turns_used': turns_used,
            'total_tokens': total_tokens, 'code_executions': code_executions,
            'n_python_calls': len(code_executions), 'n_python_errors': sum(1 for c in code_executions if c['is_error'])}

In [ ]:
def generate_all_traces(df):
    os.makedirs(CFG.output_dir, exist_ok=True)
    
    config = {'model': CFG.model_name, 'n_samples': CFG.n_samples, 'max_turns': CFG.max_turns,
              'n_problems': len(df), 'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}
    with open(f"{CFG.output_dir}/config.json", 'w') as f: json.dump(config, f, indent=2)
    
    all_results, total_correct, start_time = [], 0, time.time()
    
    for idx, row in df.iterrows():
        prob_id = str(row['id'])
        problem_text = row['problem']
        ground_truth = int(row['answer']) if pd.notna(row.get('answer')) else None
        
        print(f"\n{'='*60}")
        print(f"Problem {idx+1}/{len(df)} [{prob_id}] (GT={ground_truth})")
        
        attempts = []
        for i in range(CFG.n_samples):
            seed = CFG.seed + idx * 100 + i * 7
            t0 = time.time()
            
            try:
                result = solve_once(problem_text, seed)
                result.update({'attempt_idx': i, 'seed': seed, 'wall_time_s': round(time.time()-t0, 2)})
            except Exception as e:
                result = {'attempt_idx': i, 'answer': None, 'entropy': float('inf'), 'error': str(e)[:200]}
            
            attempts.append(result)
            print(f"  {i+1}/{CFG.n_samples}: ans={result.get('answer')}, ent={result.get('entropy', 0):.3f}")
        
        valid = [a['answer'] for a in attempts if a['answer'] is not None]
        default_answer = Counter(valid).most_common(1)[0][0] if valid else 0
        is_correct = default_answer == ground_truth if ground_truth is not None else None
        if is_correct: total_correct += 1
        
        print(f"  >> {'✓' if is_correct else '✗'} Default={default_answer}, Votes={dict(Counter(valid))}")
        
        trace = {'problem_id': prob_id, 'problem_text': problem_text, 'ground_truth': ground_truth,
                 'wall_time_s': round(time.time() - t0, 2), 'attempts': attempts,
                 'default_answer': default_answer, 'default_method': 'majority_vote', 'default_votes': dict(Counter(valid))}
        
        with open(f"{CFG.output_dir}/problem_{prob_id}.json", 'w') as f:
            json.dump(trace, f, indent=2, default=lambda x: str(x) if isinstance(x, float) and math.isinf(x) else x)
        
        all_results.append({'problem_id': prob_id, 'correct': is_correct, 'ground_truth': ground_truth})
        
        elapsed = time.time() - start_time
        print(f"  Progress: {idx+1}/{len(df)} | {total_correct} correct | {elapsed/60:.1f}min elapsed")
    
    total = sum(1 for r in all_results if r['correct'] is not None)
    accuracy = total_correct / total if total else 0
    
    summary = {'model': CFG.model_name, 'correct': total_correct, 'total': total, 'accuracy': round(accuracy, 4),
               'total_time_s': round(time.time() - start_time, 1), 'per_problem': all_results}
    with open(f"{CFG.output_dir}/summary.json", 'w') as f: json.dump(summary, f, indent=2)
    
    print(f"\n{'#'*60}")
    print(f"COMPLETE: {total_correct}/{total} ({accuracy*100:.1f}%)")
    print(f"Time: {(time.time()-start_time)/60:.1f} minutes")
    return summary

summary = generate_all_traces(df)

In [ ]:
print(f"\n{'='*60}")
print(f"AIME Trace Validation Results ({CFG.model_name})")
print(f"{'='*60}")
print(f"Accuracy: {summary['accuracy']*100:.1f}%")
print(f"Correct: {summary['correct']}/{summary['total']}")
print(f"Time: {summary['total_time_s']/60:.1f} minutes")
print(f"\nTraces saved to: {CFG.output_dir}")